# Logování — monitoring aplikací v Pythonu

Základem vývoje zdravé aplikace je správné nastavení **monitoringu**. Jeho základem je pro nás **logování** — systematické zaznamenávání událostí, které se v aplikaci dějí.

---

## Proč logovat?

| Důvod | Příklad |
|---|---|
| **Hledání chyb** | Díky logům zjistíme, co se stalo těsně před pádem aplikace |
| **Analýza využívání** | Kolik uživatelů používá danou funkci? |
| **Kontrola pohybu dat** | Někdy vyžadována zákony (GDPR, státní regulace) |
| **Diagnostika výkonu** | Jak dlouho trvá zpracování požadavku? |
| **Bezpečnost** | Kdo se pokusil přihlásit? Z jaké IP? |

## Hlavní části logovacího procesu

| Část | Zodpovědnost |
|---|---|
| **Logger** | Objekt, přes který v kódu **zapisujeme** zprávy |
| **Handler** | Určuje **kam** se zprávy posílají (soubor, konzole, síť…) |
| **Formatter** | Určuje **formát** zprávy (čas, úroveň, text…) |
| **Filter** | Umožňuje **filtrovat**, které zprávy se zaznamenají |

```
Logger  ──►  Filter  ──►  Handler  ──►  Formatter  ──►  Výstup (soubor/konzole/...)
```

## Úrovně závažnosti (log levels)

Každá zpráva má svou **úroveň závažnosti** — od nejméně po nejvíce kritickou:

| Úroveň | Číslo | Kdy použít | Příklad |
|---|---|---|---|
| `DEBUG` | 10 | Detailní informace pro vývojáře | `Proměnná x = 42` |
| `INFO` | 20 | Potvrzení, že vše funguje správně | `Server spuštěn na portu 8080` |
| `WARNING` | 30 | Něco neočekávaného, ale aplikace běží | `Disk je zaplněn na 90 %` |
| `ERROR` | 40 | Chyba — něco se nezdařilo | `Nepodařilo se připojit k DB` |
| `CRITICAL` | 50 | Fatální chyba — aplikace nemůže pokračovat | `Systém spadl, data ztracena` |

> **Tip:** Logger zobrazí pouze zprávy na své úrovni **a výš**. Logger s `level=WARNING` zobrazí WARNING, ERROR a CRITICAL, ale ne DEBUG a INFO.

## Základní pravidla logování

1. **Používat logovací knihovnu** — v Pythonu vestavěný modul `logging`, nikdy ne `print()`
2. **Logovat na správné úrovni** — ulehčuje filtrování a analýzu
3. **Psát užitečné zprávy** — co se stalo, s jakými hodnotami, pro koho
4. **Používat angličtinu** — zamezuje problémům s kódováním (ASCII) a jazykovou bariérou
5. **Nelogovat citlivé údaje** — hesla, tokeny, osobní údaje
6. **Logovat do více cílů** — konzole pro vývoj, soubor pro produkci

---

# Modul `logging` — základní použití

Python má vestavěný modul `logging` — **nemusíme nic instalovat**.

## Nejjednodušší použití — `basicConfig`

Funkce `logging.basicConfig()` nastaví výchozí logger, handler i formatter jedním voláním:

In [ ]:
import logging

# Nastavení základního logování — výstup do konzole
logging.basicConfig(
    level=logging.DEBUG,  # minimální úroveň zpráv
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Zkusíme všechny úrovně
logging.debug("Toto je DEBUG zpráva")
logging.info("Toto je INFO zpráva")
logging.warning("Toto je WARNING zpráva")
logging.error("Toto je ERROR zpráva")
logging.critical("Toto je CRITICAL zpráva")

> **⚠️ Pozor:** `basicConfig()` funguje pouze jednou (při prvním volání). V Jupyter Notebooku se může stát, že opakované spouštění buněk neprovede změny. V takovém případě restartujte kernel.

> **Pozn.:** Výchozí úroveň (bez `basicConfig`) je `WARNING` — zprávy `DEBUG` a `INFO` se **nezobrazí**.

---

# Logger — vytváření pojmenovaných loggerů

V praxi nepoužíváme přímo `logging.info(...)`, ale vytvoříme si **pojmenovaný logger**. To nám umožní:

- Rozlišit, **z jaké části** aplikace zpráva pochází
- Nastavit **různé úrovně** pro různé části
- Přidat **různé handlery** (soubor, konzole…)

## Syntaxe

```python
logger = logging.getLogger("nazev_loggeru")
```

> **Tip:** Jako název se často používá `__name__` — automaticky se doplní název modulu/souboru.

## Hierarchie loggerů

Loggery tvoří **stromovou hierarchii** oddělenou tečkami. Kořenový (root) logger je nadřazený všem:

```
root
├── app
│   ├── app.database
│   └── app.auth
└── utils
```

Zprávy z podřízeného loggeru **probublají nahoru** k nadřazenému (pokud to nezakážeme).

### Příklad

In [ ]:
import logging

# Kořenový logger
root_logger = logging.getLogger()

# Pojmenované loggery v hierarchii
app_logger = logging.getLogger("app")
db_logger = logging.getLogger("app.database")
auth_logger = logging.getLogger("app.auth")

# Přidáme handler na root logger (všechny zprávy se sem dostanou)
console = logging.StreamHandler()
console.setFormatter(logging.Formatter("%(name)s - %(levelname)s - %(message)s"))
root_logger.addHandler(console)
root_logger.setLevel(logging.DEBUG)

# Zprávy z různých loggerů
root_logger.info("Zpráva z root loggeru")
app_logger.info("Zpráva z app loggeru")
db_logger.warning("Zpráva z app.database loggeru")
auth_logger.error("Zpráva z app.auth loggeru")

# ☝️ Všechny zprávy se zobrazí, protože probublají do root loggeru

---

# Handler — kam se zprávy posílají

Handler určuje **destinaci** logů. Jeden logger může mít **více handlerů** (např. současně konzole + soubor).

## Nejčastější typy handlerů

| Handler | Kam zapisuje | Import |
|---|---|---|
| `StreamHandler` | Konzole (stdout/stderr) | `logging.StreamHandler()` |
| `FileHandler` | Soubor | `logging.FileHandler('app.log')` |
| `RotatingFileHandler` | Soubor s **rotací** (max. velikost) | `logging.handlers.RotatingFileHandler(...)` |
| `TimedRotatingFileHandler` | Soubor s **časovou rotací** (denně, týdně…) | `logging.handlers.TimedRotatingFileHandler(...)` |

> **Tip:** Rotace souborů je důležitá v produkci — bez ní by log soubor neomezeně rostl.

### Příklad — StreamHandler (konzole)

In [ ]:
import logging

logger = logging.getLogger("ukazka_stream")
logger.setLevel(logging.DEBUG)

# Handler pro výstup do konzole
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)

# Přiřadíme handler k loggeru
logger.addHandler(console_handler)

logger.info("Tato zpráva se zobrazí v konzoli")
logger.debug("A tato také")

### Příklad — FileHandler (soubor)

In [ ]:
import logging

logger = logging.getLogger("ukazka_file")
logger.setLevel(logging.DEBUG)

# Handler pro zápis do souboru
# mode="w" → přepíše soubor při každém spuštění
# mode="a" → připojí na konec (výchozí)
file_handler = logging.FileHandler("ukazka.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.INFO)  # do souboru zapíše jen INFO a výš

logger.addHandler(file_handler)

logger.debug("DEBUG — toto se do souboru NEZAPÍŠE (handler má level INFO)")
logger.info("INFO — toto se do souboru zapíše")
logger.warning("WARNING — toto se do souboru zapíše")

print("✅ Zkontrolujte soubor 'ukazka.log' ve stejné složce.")

---

# Formatter — formát zprávy

Formatter určuje, **jak bude zpráva vypadat**. Definujeme ho jako textový vzor s proměnnými.

## Nejčastější proměnné formátovacího řetězce

| Proměnná | Popis | Příklad výstupu |
|---|---|---|
| `%(asctime)s` | Datum a čas | `2025-10-10 09:15:21,362` |
| `%(name)s` | Název loggeru | `app.database` |
| `%(levelname)s` | Úroveň zprávy | `WARNING` |
| `%(message)s` | Text zprávy | `Disk plný` |
| `%(filename)s` | Název souboru | `main.py` |
| `%(funcName)s` | Název funkce | `process_data` |
| `%(lineno)d` | Číslo řádku | `42` |
| `%(module)s` | Název modulu | `main` |
| `%(process)d` | ID procesu | `12345` |
| `%(thread)d` | ID vlákna | `140735` |

### Příklad

In [ ]:
import logging

logger = logging.getLogger("ukazka_formatter")
logger.setLevel(logging.DEBUG)

# Handler s formatterem
handler = logging.StreamHandler()
formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"  # vlastní formát data
)
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.debug("Detailní zpráva")
logger.info("Vše běží správně")
logger.warning("Pozor, něco je podezřelé")
logger.error("Nastala chyba")

---

# Kombinace — konzole + soubor

V praxi chceme obvykle logovat **současně do konzole i do souboru**, často s **různými úrovněmi**:

- **Konzole** → `DEBUG` a výše (pro vývojáře při vývoji)
- **Soubor** → `WARNING` a výše (pro produkční analýzu)

### Příklad

In [ ]:
import logging

logger = logging.getLogger("moje_aplikace")
logger.setLevel(logging.DEBUG)  # logger propouští vše od DEBUG

# Handler 1: konzole — zobrazí vše (DEBUG+)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_formatter = logging.Formatter("%(levelname)-8s | %(message)s")
console_handler.setFormatter(console_formatter)

# Handler 2: soubor — zapíše jen WARNING+
file_handler = logging.FileHandler("aplikace.log", mode="w", encoding="utf-8")
file_handler.setLevel(logging.WARNING)
file_formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(file_formatter)

# Přidáme oba handlery
logger.addHandler(console_handler)
logger.addHandler(file_handler)

# Testovací zprávy
logger.debug("Debug — pouze konzole")
logger.info("Info — pouze konzole")
logger.warning("Warning — konzole + soubor")
logger.error("Error — konzole + soubor")

print("\n✅ Zkontrolujte soubor 'aplikace.log' — najdete tam jen WARNING a ERROR.")

---

# Filter — pokročilé filtrování

Kromě úrovně lze zprávy filtrovat i podle **vlastních pravidel** pomocí třídy `logging.Filter` nebo vlastní funkce.

### Příklad — filtr podle názvu loggeru

In [ ]:
import logging

# Filtr propustí pouze zprávy z loggerů začínajících na "app.database"
db_filter = logging.Filter("app.database")

handler = logging.StreamHandler()
handler.addFilter(db_filter)
handler.setFormatter(logging.Formatter("%(name)s - %(message)s"))

root = logging.getLogger()
root.addHandler(handler)
root.setLevel(logging.DEBUG)

# Pouze zprávy z app.database projdou filtrem
logging.getLogger("app.database").info("SELECT * FROM users")     # ✅ projde
logging.getLogger("app.auth").info("User logged in")              # ❌ neprojde
logging.getLogger("app.database.pool").info("Connection opened")  # ✅ projde (podřízený)

### Příklad — vlastní filtr jako funkce

In [ ]:
import logging

# Vlastní filtr — propustí jen zprávy obsahující slovo "important"
def only_important(record):
    return "important" in record.getMessage().lower()

logger = logging.getLogger("filtr_demo")
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.addFilter(only_important)
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

logger.info("This is an important message")   # ✅ projde
logger.info("This is a regular message")       # ❌ neprojde
logger.warning("IMPORTANT: disk almost full")  # ✅ projde

---

# Logování výjimek (exception logging)

Při zachycení výjimky je užitečné zalogovat celý **traceback** (výpis zásobníku). K tomu slouží metoda `logger.exception()` nebo parametr `exc_info=True`.

### Příklad

In [ ]:
import logging

logger = logging.getLogger("exception_demo")
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(handler)

def deleni(a, b):
    try:
        vysledek = a / b
        logger.info(f"{a} / {b} = {vysledek}")
        return vysledek
    except ZeroDivisionError:
        # logger.exception() automaticky zaloguje traceback na úrovni ERROR
        logger.exception(f"Chyba při dělení {a} / {b}")
        return None

deleni(10, 2)  # ✅ OK
deleni(10, 0)  # ❌ Zaloguje chybu + traceback

---

# Praktický příklad — logování ve třídě

Ukážeme si, jak vypadá logování v reálnějším scénáři — třída `Hrac` s dvěma loggery (INFO + DEBUG), které zapisují do společného souboru.

### Příklad

In [ ]:
import logging

# Společný handler — zápis do souboru
file_handler = logging.FileHandler("logovani.txt", mode="w", encoding="utf-8")
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)

# Logger pro INFO zprávy (běžné události)
logger_info = logging.getLogger("info_logger")
logger_info.setLevel(logging.INFO)
logger_info.addHandler(file_handler)

# Logger pro DEBUG zprávy (podrobnosti pro vývojáře)
logger_debug = logging.getLogger("debug_logger")
logger_debug.setLevel(logging.DEBUG)
logger_debug.addHandler(file_handler)


class Hrac:
    def __init__(self, jmeno):
        self.jmeno = jmeno
        self.skore = 0
        logger_info.info(f"Hrac {self.jmeno} byl vytvoren.")
        logger_debug.debug(f"Vytvoren objekt Hrac(jmeno='{self.jmeno}', skore={self.skore})")

    def pridat_body(self, body):
        stary_stav = self.skore
        self.skore += body
        logger_info.info(f"Hrac {self.jmeno} ziskal {body} bodu. Celkem ma {self.skore}.")
        logger_debug.debug(f"Zmena skore: {stary_stav} -> {self.skore}")

    def zkontrolovat_vyhru(self, limit):
        if self.skore >= limit:
            logger_info.info(f"Hrac {self.jmeno} vyhral!")
            logger_debug.debug(f"Kontrola vyhry: skore={self.skore}, limit={limit} => VYHRAL")
            return True
        else:
            logger_debug.debug(f"Kontrola vyhry: skore={self.skore}, limit={limit} => NEVYHRAL")
            return False


# Ukázka použití
hrac1 = Hrac("Petr")
hrac1.pridat_body(5)
hrac1.pridat_body(10)
hrac1.zkontrolovat_vyhru(10)

print("✅ Zkontrolujte soubor 'logovani.txt' — najdete tam všechny záznamy.")

### Ukázkový obsah souboru `logovani.txt`

```
2025-10-10 09:15:21,362 - info_logger - INFO - Hrac Petr byl vytvoren.
2025-10-10 09:15:21,364 - debug_logger - DEBUG - Vytvoren objekt Hrac(jmeno='Petr', skore=0)
2025-10-10 09:15:21,365 - info_logger - INFO - Hrac Petr ziskal 5 bodu. Celkem ma 5.
2025-10-10 09:15:21,367 - debug_logger - DEBUG - Zmena skore: 0 -> 5
2025-10-10 09:15:21,368 - info_logger - INFO - Hrac Petr ziskal 10 bodu. Celkem ma 15.
2025-10-10 09:15:21,369 - debug_logger - DEBUG - Zmena skore: 5 -> 15
2025-10-10 09:15:21,371 - info_logger - INFO - Hrac Petr vyhral!
2025-10-10 09:15:21,372 - debug_logger - DEBUG - Kontrola vyhry: skore=15, limit=10 => VYHRAL
```

---

# RotatingFileHandler — rotace log souborů

V produkci log soubory neustále rostou. `RotatingFileHandler` automaticky **rotuje** (přejmenuje a vytvoří nový), když soubor překročí zadanou velikost.

```python
from logging.handlers import RotatingFileHandler

handler = RotatingFileHandler(
    "app.log",
    maxBytes=1_000_000,  # 1 MB
    backupCount=5        # uchová 5 starých souborů (app.log.1, app.log.2, …)
)
```

### Příklad

In [ ]:
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger("rotace_demo")
logger.setLevel(logging.DEBUG)

# Rotace při 5 KB, max 3 zálohy
handler = RotatingFileHandler(
    "rotace_demo.log",
    maxBytes=5_000,
    backupCount=3,
    encoding="utf-8"
)
handler.setFormatter(logging.Formatter("%(asctime)s - %(message)s"))
logger.addHandler(handler)

# Zapíšeme hodně zpráv, aby se soubor zrotoval
for i in range(200):
    logger.info(f"Zprava cislo {i}: Lorem ipsum dolor sit amet")

print("✅ Ve složce by měly být soubory: rotace_demo.log, .log.1, .log.2, .log.3")

---

# Logování vs. print()

Proč nepoužívat `print()` pro ladění?

| | `print()` | `logging` |
|---|---|---|
| Úrovně závažnosti | ❌ Ne | ✅ DEBUG, INFO, WARNING, ERROR, CRITICAL |
| Výstup do souboru | ❌ Musíte řešit ručně | ✅ FileHandler |
| Formátování (čas, modul…) | ❌ Musíte psát ručně | ✅ Formatter |
| Filtrování | ❌ Nelze | ✅ Level + Filter |
| Vypnutí v produkci | ❌ Musíte mazat/komentovat | ✅ Změna úrovně |
| Více výstupů najednou | ❌ Ne | ✅ Více handlerů |
| Traceback výjimek | ❌ Ručně | ✅ `logger.exception()` |

> **Závěr:** `print()` je pro rychlé ladění. V jakékoli aplikaci, která poběží déle než pár minut, vždy používejte `logging`.

---

# Shrnutí

| Koncept | Co dělá | Příkaz |
|---|---|---|
| Logger | Vytváří zprávy | `logging.getLogger("nazev")` |
| Handler | Posílá zprávy na výstup | `StreamHandler()`, `FileHandler()` |
| Formatter | Formátuje zprávy | `Formatter("%(asctime)s - %(message)s")` |
| Filter | Filtruje zprávy | `Filter("app.database")` |
| Level | Minimální úroveň | `logger.setLevel(logging.WARNING)` |
| Exception | Loguje traceback | `logger.exception("Chyba")` |
| Rotace | Omezuje velikost logu | `RotatingFileHandler(maxBytes=...)` |

### Typická konfigurace pro reálnou aplikaci

```python
import logging
from logging.handlers import RotatingFileHandler

# Vytvoření loggeru
logger = logging.getLogger("moje_app")
logger.setLevel(logging.DEBUG)

# Konzole — DEBUG+ (pro vývoj)
console = logging.StreamHandler()
console.setLevel(logging.DEBUG)
console.setFormatter(logging.Formatter("%(levelname)-8s | %(message)s"))

# Soubor — WARNING+ (pro produkci), rotace 5 MB, 3 zálohy
file = RotatingFileHandler("app.log", maxBytes=5_000_000, backupCount=3)
file.setLevel(logging.WARNING)
file.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))

logger.addHandler(console)
logger.addHandler(file)
```

---

# Cvičení

> V každém cvičení pracujte pouze s modulem `logging` (případně `logging.handlers`). Nemusíte nic instalovat.

## Cvičení 1 — Základní logování

1. Vytvořte logger s názvem `"cviceni1"`.
2. Nastavte úroveň na `DEBUG`.
3. Přidejte `StreamHandler` s formátem: viz `očekávaný výstup`
4. Vypište jednu zprávu na každé úrovni: DEBUG, INFO, WARNING, ERROR, CRITICAL.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
DEBUG: Toto je debug
INFO: Toto je info
WARNING: Toto je warning
ERROR: Toto je error
CRITICAL: Toto je critical
```

</details>

In [ ]:
import logging

# TODO: Vytvořte logger, handler, formatter a vypište zprávy →

## Cvičení 2 — Logování do souboru

1. Vytvořte logger s názvem `"cviceni2"`.
2. Přidejte `FileHandler`, který zapíše logy do souboru `cviceni2.log` (s `mode="w"`).
3. Nastavte formát: viz `očekávaný výstup`
4. Zalogujte 3 zprávy různých úrovní.
5. Na konci otevřete a vypište obsah souboru pomocí `open()` + `read()`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
Obsah souboru cviceni2.log:
2025-10-10 12:00:00,000 | INFO | Server started
2025-10-10 12:00:00,001 | WARNING | High memory usage
2025-10-10 12:00:00,002 | ERROR | Database connection lost
```

</details>

In [ ]:
import logging

# TODO: Logger + FileHandler + 3 zprávy + výpis obsahu souboru →

## Cvičení 3 — Dva handlery najednou

1. Vytvořte logger s názvem `"cviceni3"` a úrovní `DEBUG`.
2. Přidejte **dva handlery**:
   - `StreamHandler` s úrovní `DEBUG` (zobrazí vše v konzoli)
   - `FileHandler` s úrovní `ERROR` (do souboru `cviceni3.log` zapíše jen ERROR+)
3. Oba handlery mají mít vlastní formatter.
4. Zalogujte zprávy: DEBUG, INFO, WARNING, ERROR.
5. Ověřte, že v souboru je pouze ERROR.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
--- Konzole ---
DEBUG: Debug zprava
INFO: Info zprava
WARNING: Warning zprava
ERROR: Error zprava

--- Obsah souboru ---
2025-... - cviceni3 - ERROR - Error zprava
```

</details>

In [ ]:
import logging

# TODO: Logger + StreamHandler (DEBUG) + FileHandler (ERROR) →

## Cvičení 4 — Logování výjimek

1. Vytvořte logger `"cviceni4"` s `StreamHandler`.
2. Napište funkci `nacti_cislo(text)`, která:
   - Zkusí převést `text` na `int`
   - Pokud se to podaří, zaloguje `INFO` se zprávou: `"Nacteno cislo: X"`
   - Pokud dojde k `ValueError`, zaloguje chybu pomocí `logger.exception()`
3. Zavolejte funkci s hodnotami `"42"`, `"abc"`, `"100"`.

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
INFO: Nacteno cislo: 42
ERROR: Chyba pri konverzi: abc
Traceback (most recent call last):
  ...
ValueError: invalid literal for int() with base 10: 'abc'
INFO: Nacteno cislo: 100
```

</details>

In [ ]:
import logging

# TODO: Logger + funkce nacti_cislo s exception loggingem →

## Cvičení 5 — Kompletní úloha

Vytvořte třídu `BankovniUcet` s atributy `majitel` (str) a `zustatek` (float, výchozí 0).

Metody:
- `vlozit(castka)` — přidá peníze, zaloguje INFO
- `vybrat(castka)` — odečte peníze; pokud není dostatek, zaloguje WARNING a peníze neodečte
- `prevod(cilovy_ucet, castka)` — provede převod, zaloguje INFO; při nedostatku zaloguje ERROR

Konfigurace logování:
- Logger s názvem `"banka"`
- `StreamHandler` (konzole) s úrovní `DEBUG`
- `FileHandler` (`banka.log`) s úrovní `INFO`

Testovací scénář:
1. Vytvořte 2 účty (Alice: 1000, Bob: 500)
2. Alice vloží 500
3. Bob se pokusí vybrat 1000 (nedostatek!)
4. Alice převede 200 na Boba
5. Vypište konečné zůstatky

<details>
<summary>🔎 Očekávaný výstup (klikni pro zobrazení)</summary>

```
INFO: Ucet vytvoren: Alice, zustatek=1000
INFO: Ucet vytvoren: Bob, zustatek=500
INFO: Alice: vklad 500, novy zustatek=1500
WARNING: Bob: nedostatek prostredku pro vyber 1000 (zustatek=500)
INFO: Prevod 200 z Alice na Bob
DEBUG: Alice: zustatek 1500 -> 1300
DEBUG: Bob: zustatek 500 -> 700

Konecne zustatky: Alice=1300, Bob=700
```

</details>

In [ ]:
import logging

# TODO: Celý úkol je na vás →